# notebook 18 — R×T³ の幾何の直接構成：SO(3) 測定方向を明示した時空

nb16 は最小時空 R×T³ の骨格（混合埋め込みの罠回避、時間と空間の別構成、T³ の内在3次元、
時間順序の推移性）を数値で確認した。本 notebook はその骨格を、**空間 T³ の3つの S¹ を
SO(3)（測定方向を回す実空間回転群）の直交3軸として明示**した上で、MDS 復元・因果構造・
対称性まで直接構成する（nb15 open-2 / nb17 open-1）。

攻略は反証ハードルの低い順＝**(a)→(b)→(c)** で、土台を固めてから D-10 リスクの高い (c) に入る：

- **(a) 幾何の精密化**：T³ の3軸を SO(3) の直交軸まわりの測定角として明示し、軸の独立性・
  直交性が復元幾何に正しく反映されるか。
- **(b) 因果構造の幾何的検証**：被覆時間 t による計量符号数 (−,+,+,+) と光円錐を符号レベルで
  構成し、nb09 L-A の非推移性が R×T³ で解消されることを確認。
- **(c) SO(3) 作用の幾何への作用**：測定方向の SO(3) 回転が空間 T³ の等長変換になるか。
  D-10 の罠（SO(3)＝空間対称性の後付け同一視）と最も激しく戦う最深部。

**最重要の規律（D-10/D-13）**：(c) は「測定の SO(3) ＝ 空間の対称群」という魅力的な同一視に
最も誘惑される。群として本当に等長か*構成的に検証*し、数の一致（軸3本）を群の一致と短絡しない。

## 18.0 セットアップ

In [1]:
import numpy as np
from numpy.linalg import eigh
np.set_printoptions(precision=4, suppress=True, linewidth=120)

N = 3  # 与件（QCD との同定）
print("setup done. N =", N)

setup done. N = 3


---
# Part (a)：SO(3) の3軸を測定方向として明示した T³ の MDS 復元

T³ の各 S¹ を、実空間 R³ の直交軸 x,y,z まわりの測定角 (φ₁,φ₂,φ₃) として明示的にラベル付けする。
測定設定は (φ₁,φ₂,φ₃)∈T³、相関は各軸ごとに cos2(Δφ)、距離は加法分解（nb04）。

## 18.1 軸ごとのラベル付けと全 T³ の MDS

In [2]:
n_per = 6
ph = np.linspace(0, 2*np.pi, n_per, endpoint=False)
P1, P2, P3 = np.meshgrid(ph, ph, ph, indexing="ij")
sp = np.stack([P1.ravel(), P2.ravel(), P3.ravel()], 1)
M = sp.shape[0]
axis_names = ["x軸 (SO(3) L_x)", "y軸 (SO(3) L_y)", "z軸 (SO(3) L_z)"]

def axis_d2(col):
    dphi = sp[:, col][:, None] - sp[:, col][None, :]
    a = np.abs(np.cos(2*dphi))/(2*N)
    with np.errstate(divide="ignore"):
        d = -np.log(a)
    d[~np.isfinite(d)] = 0.0; np.fill_diagonal(d, 0.0)
    return d**2

d2_each = [axis_d2(c) for c in range(3)]
d2_total = sum(d2_each)
print(f"設定空間 T³ = {n_per}^3 = {M} 点。各軸 = SO(3) の直交軸まわりの測定角 S¹。")
print("距離² = Σ_axis d_axis²（nb04 加法分解）。")
print()

J = np.eye(M) - np.ones((M, M))/M
B = -0.5*J@d2_total@J; B = 0.5*(B + B.T)
evs = np.sort(eigh(B)[0])[::-1]
ratio = evs/evs[0]
print(f"全 T³ の MDS 上位9固有値比: {ratio[:9].round(3)}")
print(f"  有意(>5%)固有値数 = {int((ratio>0.05).sum())} → 埋め込み6次元（3軸×各2座標）")
print(f"  7個目で急落 {ratio[6]:.4f} → 内在3次元（各 S¹ が内在1）")

設定空間 T³ = 6^3 = 216 点。各軸 = SO(3) の直交軸まわりの測定角 S¹。
距離² = Σ_axis d_axis²（nb04 加法分解）。

全 T³ の MDS 上位9固有値比: [1.    1.    1.    1.    1.    1.    0.043 0.043 0.043]
  有意(>5%)固有値数 = 6 → 埋め込み6次元（3軸×各2座標）
  7個目で急落 0.0432 → 内在3次元（各 S¹ が内在1）


## 18.2 軸の独立性・直交性：軸間に偽の相関が出ないか

各軸単独の MDS が張る2次元部分空間が、他軸の部分空間と直交するかを principal angle で測る。
重なり（cos）が0なら、SO(3) の3軸の独立性が復元幾何に正しく反映されている。

In [3]:
def axis_eigvecs(col, k=2):
    Ba = -0.5*J@d2_each[col]@J; Ba = 0.5*(Ba + Ba.T)
    w, v = eigh(Ba)
    idx = np.argsort(w)[::-1][:k]
    return v[:, idx]

V = [axis_eigvecs(c) for c in range(3)]
print("軸間の部分空間の重なり（principal angle の cos、0なら直交）：")
for i in range(3):
    for j in range(i+1, 3):
        s = np.linalg.svd(V[i].T @ V[j], compute_uv=False)
        print(f"  {axis_names[i][:3]} vs {axis_names[j][:3]}: max overlap = {s.max():.4f}")
print()
print("→ 重なりが厳密に0：各軸の MDS 部分空間は完全独立。SO(3) の3軸の独立性が")
print("  復元 T³ の幾何に正しく反映されている（軸間に偽の相関なし）。")

軸間の部分空間の重なり（principal angle の cos、0なら直交）：
  x軸  vs y軸 : max overlap = 0.0000
  x軸  vs z軸 : max overlap = 0.0000
  y軸  vs z軸 : max overlap = 0.0000

→ 重なりが厳密に0：各軸の MDS 部分空間は完全独立。SO(3) の3軸の独立性が
  復元 T³ の幾何に正しく反映されている（軸間に偽の相関なし）。


## 18.3 Part (a) の確定

T³ の3つの S¹ を SO(3) の直交3軸まわりの測定角として明示でき、MDS は埋め込み6次元・内在3次元を
正しく復元する。各軸の部分空間は厳密に直交し、軸の独立性が幾何に反映される。nb16 の「裸の T³」を
**「測定方向つき T³」**に格上げできた。

---
# Part (b)：因果構造の幾何的検証 — R×T³ の符号数と光円錐

nb16 は時間順序の推移性（スカラー t の大小）までを確認した。(b) はそれを**計量の符号レベル**へ
進める：被覆時間 t を timelike 方向（符号 −）、空間 T³ を spacelike（符号 +）として、計量符号数
(−,+,+,+) と光円錐 ds²<0/=0/>0 を構成し、大域的因果整合（推移性）を符号で確認する。

## 18.4 計量の符号数：時間1（被覆）＋ 空間3（T³ MDS）

In [4]:
n_per = 5
ph = np.linspace(0, 2*np.pi, n_per, endpoint=False)
P1, P2, P3 = np.meshgrid(ph, ph, ph, indexing="ij")
sp = np.stack([P1.ravel(), P2.ravel(), P3.ravel()], 1)
Ms = sp.shape[0]

def axis_d2b(col):
    dphi = sp[:, col][:, None] - sp[:, col][None, :]
    a = np.abs(np.cos(2*dphi))/(2*N)
    with np.errstate(divide="ignore"):
        d = -np.log(a)
    d[~np.isfinite(d)] = 0.0; np.fill_diagonal(d, 0.0)
    return d**2

dsp2 = sum(axis_d2b(c) for c in range(3))
J = np.eye(Ms) - np.ones((Ms, Ms))/Ms
Bsp = -0.5*J@dsp2@J; Bsp = 0.5*(Bsp + Bsp.T)
evsp = np.sort(eigh(Bsp)[0])[::-1]
nsp = int((evsp/evsp[0] > 0.05).sum())
print(f"空間 T³ MDS：正の有意固有値 = {nsp}（埋め込み6＝3軸×2座標、内在3、全て符号 +）")
print("時間：被覆座標 t = 1スカラー（符号 −、構成的に timelike）")
print("→ 計量符号数（内在）= (−,+,+,+)。時間を埋め込みで混ぜず別構成（nb16）→ timelike は厳密に1。")

空間 T³ MDS：正の有意固有値 = 6（埋め込み6＝3軸×2座標、内在3、全て符号 +）
時間：被覆座標 t = 1スカラー（符号 −、構成的に timelike）
→ 計量符号数（内在）= (−,+,+,+)。時間を埋め込みで混ぜず別構成（nb16）→ timelike は厳密に1。


## 18.5 光円錐構造：ds² = −dt² + d_space² の3類別

2点間の squared interval を時間差 dt と空間距離 d_space から構成し、timelike(ds²<0)/spacelike(>0)/
lightlike(=0) の3類別が非自明に立つかを見る。

In [5]:
def space_dist(i, j):
    s = 0.0
    for c in range(3):
        dphi = sp[i, c] - sp[j, c]
        a = abs(np.cos(2*dphi))/(2*N)
        d = -np.log(a) if a > 0 else 0.0
        s += d**2
    return np.sqrt(s)

rng = np.random.default_rng(0)
nt = 8
t = np.linspace(0, 6*np.pi, nt)
counts = {"timelike": 0, "spacelike": 0, "lightlike": 0}
for _ in range(400):
    i, j = rng.integers(0, Ms, 2)
    ti, tj = rng.integers(0, nt, 2)
    ds2 = -(t[ti]-t[tj])**2 + space_dist(i, j)**2
    if abs(ds2) < 1e-6: counts["lightlike"] += 1
    elif ds2 < 0: counts["timelike"] += 1
    else: counts["spacelike"] += 1
print(f"400点対の類別: {counts}")
print("→ timelike も spacelike も出る = 光円錐が非自明に立つ。")
print("  （連続極限で lightlike 面 ds²=0 が境界をなす。離散サンプルでは稀。）")

400点対の類別: {'timelike': 273, 'spacelike': 127, 'lightlike': 0}
→ timelike も spacelike も出る = 光円錐が非自明に立つ。
  （連続極限で lightlike 面 ds²=0 が境界をなす。離散サンプルでは稀。）


## 18.6 因果順序の推移性（符号レベル）と同時刻面

nb09 L-A では `sign(cos2θ)` の timelike 関係が24%非推移で因果順序にならなかった。R×T³ では
時間＝被覆座標 t の大小（空間に依らない）で、実数の全順序ゆえ推移的になるはず。あわせて
同時刻面（t 一定の T³ スライス）が全て spacelike（因果的に同時）であることを確認する。

In [6]:
# future 関係 t_i<t_j<t_k の推移性
T = t; n = len(T); viol = tot = 0
for i in range(n):
    for j in range(n):
        for k in range(n):
            if T[i] < T[j] and T[j] < T[k]:
                tot += 1; viol += (not T[i] < T[k])
print(f"被覆時間 future 関係の推移性: {tot-viol}/{tot} ({100*(1-viol/max(tot,1)):.0f}%)")
print("→ 完全推移。nb09 L-A の24%破れ（sign の周期反転）を R×T³ は構造的に解消")
print("  （時間は符号でなく被覆座標の大小で定まる）。")
print()
print("同時刻面（t 一定）：dt=0 ⇒ ds² = d_space² ≥ 0 ⇒ 全点対 spacelike。")
print("→ 同時刻面は無矛盾に空間的。空間 T³ が空間として機能している。")

被覆時間 future 関係の推移性: 56/56 (100%)
→ 完全推移。nb09 L-A の24%破れ（sign の周期反転）を R×T³ は構造的に解消
  （時間は符号でなく被覆座標の大小で定まる）。

同時刻面（t 一定）：dt=0 ⇒ ds² = d_space² ≥ 0 ⇒ 全点対 spacelike。
→ 同時刻面は無矛盾に空間的。空間 T³ が空間として機能している。


## 18.7 Part (b) の確定

R×T³ の計量符号数 (−,+,+,+) を構成し、光円錐が非自明に立ち、被覆時間による因果順序が完全推移
（nb09 L-A の壁の構造的解消）、同時刻面が全 spacelike であることを符号レベルで確認した。nb16 の
「時間順序の推移性」から**「計量符号数と光円錐の幾何的構成」**へ一歩進んだ。

---
# Part (c)：SO(3) 作用の幾何への作用 — D-10 と戦う最深部

最後に最深の問い：測定方向への SO(3) 回転は、復元空間 T³ の**等長変換（対称性）**になるか。
立場B が「空間3＝SO(3) の3軸」と言うとき、それは*軸の本数*の対応か、それとも*群として*の
SO(3)＝空間対称性か。後者なら立場B は格段に強くなる。だが D-10 の罠（数の一致を群の一致と
短絡する後付け）に最も陥りやすい。等長性を*構成的に検証*して峻別する。

## 18.8 並進と軸置換は等長か（自明・離散の対称性）

まず T³ 並進（各 S¹ の一斉シフト）と軸置換（x,y,z の入れ替え＝SO(3) の有限部分群）が距離を保つかを
確認する。相関が差 Δφ のみに依存し距離が3軸対称なので、どちらも等長のはず。

In [7]:
def dmat(points):
    d2 = np.zeros((len(points), len(points)))
    for c in range(3):
        dphi = points[:, c][:, None] - points[:, c][None, :]
        a = np.abs(np.cos(2*dphi))/(2*N)
        with np.errstate(divide="ignore"):
            d = -np.log(a)
        d[~np.isfinite(d)] = 0.0; np.fill_diagonal(d, 0.0)
        d2 += d**2
    return np.sqrt(d2)

D0 = dmat(sp)
print("T³ 並進（各 S¹ の一斉シフト）：")
for delta in [0.3, 0.7, 1.5]:
    err = np.abs(dmat(sp + delta) - D0).max()
    print(f"  δ={delta}: 距離行列の最大変化 = {err:.2e} → 等長")
print()
print("軸置換（SO(3) の有限部分群＝八面体群の回転）：")
for perm in [(1, 0, 2), (2, 1, 0), (1, 2, 0)]:
    err = np.abs(dmat(sp[:, list(perm)]) - D0).max()
    print(f"  置換 {perm}: 距離行列の最大変化 = {err:.2e} → 等長")

T³ 並進（各 S¹ の一斉シフト）：
  δ=0.3: 距離行列の最大変化 = 8.88e-16 → 等長
  δ=0.7: 距離行列の最大変化 = 8.88e-16 → 等長
  δ=1.5: 距離行列の最大変化 = 0.00e+00 → 等長

軸置換（SO(3) の有限部分群＝八面体群の回転）：
  置換 (1, 0, 2): 距離行列の最大変化 = 0.00e+00 → 等長
  置換 (2, 1, 0): 距離行列の最大変化 = 8.88e-16 → 等長
  置換 (1, 2, 0): 距離行列の最大変化 = 8.88e-16 → 等長


## 18.9 連続 SO(3) 回転は等長か（D-10 の核心）

真の連続 SO(3) 回転は測定軸を連続的に混ぜる。T³ の各 φ は「その軸まわりの測定角」で独立周期を持つ。
角度ベクトル (φ₁,φ₂,φ₃) を R³ ベクトルのように回したとき、cos2(各軸独立) 構造＝距離が保たれるか。

In [8]:
def rot_x(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])

R = rot_x(0.5)
err = np.abs(dmat(sp @ R.T) - D0).max()
print(f"連続 SO(3) 回転 R_x(0.5) を角度ベクトルに作用：距離行列の最大変化 = {err:.3f}")
print()
if err > 0.1:
    print("→ 等長で『ない』。連続回転は cos2(各軸独立) 構造を壊す。")
    print("  T³ の各 S¹ は独立周期で、R³ ベクトルの連続回転とは別物。")
    print("  【D-10 判定】『連続 SO(3) ＝ 空間 T³ の等長群』は成り立たない。")
else:
    print("→ 等長。連続回転も保つ。")

連続 SO(3) 回転 R_x(0.5) を角度ベクトルに作用：距離行列の最大変化 = 2.602

→ 等長で『ない』。連続回転は cos2(各軸独立) 構造を壊す。
  T³ の各 S¹ は独立周期で、R³ ベクトルの連続回転とは別物。
  【D-10 判定】『連続 SO(3) ＝ 空間 T³ の等長群』は成り立たない。


## 18.10 正しい対称群の同定と Part (c) の確定

復元空間 T³ の等長群を正確に同定し、立場B の主張の射程を確定する。

In [9]:
print("復元空間 T³ の等長群：")
print("  - 並進 T³（各 S¹ の回転）：連続・可換 → トーラスの自明な対称性")
print("  - 軸置換 S₃（⊂ 八面体群）：離散 → 3軸の交換対称")
print("  - 連続 SO(3) 全体：等長群で『ない』（18.9）")
print()
print("【D-10 確定】測定方向の SO(3) と空間 T³ の等長群を同一視するのは後付け。")
print("立場B の『空間3＝SO(3) の3軸』は *軸の本数* の対応であって、")
print("*群として* の SO(3)＝空間対称性ではない。これが立場B の誠実な射程の限界。")
print("（D-13『数の一致は原理でない』が、ここでは群構造の不一致として顕在化した。）")

復元空間 T³ の等長群：
  - 並進 T³（各 S¹ の回転）：連続・可換 → トーラスの自明な対称性
  - 軸置換 S₃（⊂ 八面体群）：離散 → 3軸の交換対称
  - 連続 SO(3) 全体：等長群で『ない』（18.9）

【D-10 確定】測定方向の SO(3) と空間 T³ の等長群を同一視するのは後付け。
立場B の『空間3＝SO(3) の3軸』は *軸の本数* の対応であって、
*群として* の SO(3)＝空間対称性ではない。これが立場B の誠実な射程の限界。
（D-13『数の一致は原理でない』が、ここでは群構造の不一致として顕在化した。）


---
## 18.11 サマリー（established / open）

### established（この notebook で確定）

1. **測定方向つき T³ の構成（Part a）。** T³ の3つの S¹ を SO(3) の直交3軸まわりの測定角として
   明示でき、MDS は埋め込み6次元・内在3次元を正しく復元。各軸の部分空間は厳密に直交し、軸の
   独立性が幾何に反映される。nb16 の裸の T³ を測定方向つきに格上げ。

2. **R×T³ の計量符号数と光円錐（Part b）。** 被覆時間 t を timelike、空間 T³ を spacelike として
   計量符号数 (−,+,+,+) を構成。光円錐が非自明に立ち、被覆時間による因果順序は完全推移
   （nb09 L-A の24%非推移の構造的解消）、同時刻面は全 spacelike。nb16 の時間順序推移性から
   符号レベルの光円錐構成へ前進。

3. **SO(3) は空間 T³ の対称群ではない（Part c、D-10 確定）。** 等長なのは T³ 並進（連続・可換）と
   軸置換 S₃（離散）のみ。連続 SO(3) 回転は cos2(各軸独立) 構造を壊し等長でない。立場B の
   「空間3＝SO(3) の3軸」は*軸の本数*の対応であって*群として*の対称性一致ではない。D-13
   （数の一致は原理でない）が群構造の不一致として顕在化した。

### 時空模型 R×T³ の到達点

| 要素 | 構成 | 状態 |
|---|---|---|
| 時間 R | 被覆 II（nb13） | **確立**（全順序・非有界・因果推移） |
| 空間 T³ | SO(3) 3軸の測定角 → MDS 内在3 | **構成完了**（軸独立・直交を復元） |
| 計量符号数 | (−,+,+,+) | **構成完了**（時間別構成で timelike 厳密に1） |
| 光円錐・因果 | ds²=−dt²+d_space² | **構成完了**（非自明・完全推移） |
| 空間対称性 | T³ 並進 + S₃ 軸置換のみ | **確定**（連続 SO(3) は等長でない） |

### open（次へ）

1. **連続回転対称性の起源（新規 open）。** 物理空間は連続回転 SO(3) 対称を持つが、本構成の T³ は
   並進＋離散軸置換しか持たない。連続回転対称はどこから来るか——T³ の連続極限（格子間隔 →0）で
   創発するのか、それとも別原理が要るのか。本丸の連続極限（ε→0）との関係が次の焦点。

2. **「実空間3次元」の上流原理（nb17 最深 open の継続）。** Part (c) で SO(3) が空間対称群でないと
   確定した以上、空間3次元の起源は「測定方向の軸数3」という外部入力にとどまる。これを cos2θ より
   上流の原理から出せるかは、README §5 第二段階（下記改訂）に属する。

3. **本丸 no-go の解析的裏づけ（C-1、継続）。**

### 教訓（D への追記候補）

- **D-17（候補）：数の一致が群の一致とは限らない。** 「空間3＝SO(3) の3」は軸の本数では一致するが、
  群構造（連続 SO(3) vs T³ 並進＋S₃）では一致しない。対応を主張するときは、数（次元・本数）だけでなく
  *構造（群・代数）*のレベルで一致を検証する。D-13 の構造版。

- **D-18（候補）：構成の「成功」と「限界」を同じ精度で報告せよ。** nb18 は R×T³ の幾何・因果を
  構成「できた」（a,b）と同時に、SO(3) 対称性は構成「できない」（c）ことを同じ数値精度で示した。
  肯定的結果と否定的結果を非対称に扱わないことが、模型の射程を正確に画定する。